# 01_bronze_ingestion — Raw Data Ingestion
## ClinVar Medallion Pipeline · Bronze Layer

Ingests raw ClinVar variant data into the Bronze Delta table.

**What happens here:**
- Download `variant_summary.txt.gz` from NCBI over HTTPS (streamed in chunks)
- Load the raw TSV into a PySpark DataFrame — no value transformations
- Rename three columns Delta can't store, and add an `ingestion_timestamp`
- Write to Delta: `workspace.clinvar.variant_summary_raw`

**Design principle:** Bronze preserves raw fidelity. Nothing is cleaned,
filtered, or cast. If Silver or Gold produce wrong results, reprocessing
always restarts from this unmodified source.

**Run `00_setup` first.**

In [0]:
%run ./00_setup

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# DOWNLOAD variant_summary.txt.gz FROM NCBI
# ─────────────────────────────────────────────────────────────────────────────
# Fetch the raw ClinVar file from NCBI over HTTPS (programmatic ingestion).
# - stream=True / chunk_size=8192: write in 8KB chunks so memory stays flat
#   regardless of file size
# - User-Agent email: NCBI's requested courtesy for bulk access, not auth

import requests

NCBI_URL = (
    "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/"
    "variant_summary.txt.gz"
)

# Unity Catalog Volumes are accessible as a regular filesystem path
# from within Databricks notebooks
RAW_FILE_PATH = f"/Volumes/workspace/clinvar/raw_files/variant_summary.txt.gz"

headers = {
    "User-Agent": "ClinVarPipeline/1.0 (etentsidou@gmail.com; learning project)"
}

print(f"Downloading from : {NCBI_URL}")
print(f"Destination      : {RAW_FILE_PATH}")
print("Starting download — this will take a few minutes...\n")

response = requests.get(NCBI_URL, headers=headers, stream=True)
response.raise_for_status()  # raises an error immediately if the request failed

# Track progress as we write chunks
bytes_written = 0
chunk_size = 8192  # 8KB

with open(RAW_FILE_PATH, "wb") as f:
    for chunk in response.iter_content(chunk_size=chunk_size):
        if chunk:  # filter out keep-alive empty chunks
            f.write(chunk)
            bytes_written += len(chunk)

mb_written = bytes_written / (1024 * 1024)
print(f"Download complete.")
print(f"File size        : {mb_written:.1f} MB")
print(f"Location         : {RAW_FILE_PATH}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# READ RAW TSV INTO PYSPARK DATAFRAME
# ─────────────────────────────────────────────────────────────────────────────
# Read the gzipped TSV into a DataFrame (PySpark handles .gz natively).
# inferSchema samples the data to guess types — fine for Bronze; Silver
# enforces types properly. No cleaning here; only #AlleleID → AlleleID
# (the # is NCBI's header marker, not part of the name).

from pyspark.sql.functions import current_timestamp

print("Reading variant_summary.txt.gz into PySpark...")

df_raw = (
    spark.read
    .option("header", "true")        # first row is column names
    .option("inferSchema", "true")   # detect column types from data
    .option("sep", "\t")             # tab-separated
    .option("quote", "")             # disable quote handling — TSV doesn't use quotes
    .csv(RAW_FILE_PATH)
)

# Rename #AlleleID — the # is NCBI's header marker, not part of the name
df_raw = df_raw.withColumnRenamed("#AlleleID", "AlleleID")

print(f"Rows loaded      : {df_raw.count():,}")
print(f"Columns          : {len(df_raw.columns)}")
print("\nSchema:")
df_raw.printSchema()

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# SANITISE COLUMN NAMES FOR DELTA COMPATIBILITY
# ─────────────────────────────────────────────────────────────────────────────
# Rename the two columns whose names contain characters Delta rejects
# (spaces, #, /, parentheses). Values untouched — names only, still Bronze.

df_raw = df_raw \
    .withColumnRenamed("RS# (dbSNP)", "RS_dbSNP") \
    .withColumnRenamed("nsv/esv (dbVar)", "nsv_esv_dbVar")

print("Column names sanitised:")
print("  RS# (dbSNP)     → RS_dbSNP")
print("  nsv/esv (dbVar) → nsv_esv_dbVar")
print(f"\nTotal columns: {len(df_raw.columns)}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# ADD METADATA AND WRITE TO BRONZE DELTA TABLE
# ─────────────────────────────────────────────────────────────────────────────
# Add ingestion_timestamp — when the data entered OUR pipeline (audit trail),
# the only column Bronze adds. Then write to the Bronze Delta table.
# overwrite = re-running replaces the table with a fresh full snapshot
# (production with daily loads might append + dedupe instead).

from pyspark.sql.functions import current_timestamp

df_bronze = df_raw.withColumn("ingestion_timestamp", current_timestamp())

print(f"Writing to Delta table: {TABLE_BRONZE}")
print("This will take a moment for a dataset this size...\n")

(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLE_BRONZE)
)

print(f"Bronze table written successfully.")
print(f"Table            : {TABLE_BRONZE}")
print(f"Rows             : {df_bronze.count():,}")
print(f"Columns          : {len(df_bronze.columns)} (including ingestion_timestamp)")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# BRONZE VALIDATION
# ─────────────────────────────────────────────────────────────────────────────
# Read back from the Delta table (not the in-memory DataFrame) to prove the
# data actually persisted and is queryable. Checks: row/column count,
# ingestion_timestamp populated, the GRCh37/38 split Silver will filter,
# a sample of messy ClinicalSignificance values, and null rates.

print("=" * 60)
print("BRONZE VALIDATION")
print("=" * 60)

df_check = spark.table(TABLE_BRONZE)

# Row and column count
print(f"\n  Rows    : {df_check.count():,}")
print(f"  Columns : {len(df_check.columns)}")

# Confirm ingestion_timestamp exists and has values
from pyspark.sql.functions import count, when, col, min, max

ts_check = df_check.select(
    count("ingestion_timestamp").alias("non_null_count"),
    min("ingestion_timestamp").alias("earliest"),
    max("ingestion_timestamp").alias("latest")
).collect()[0]

print(f"\n  ingestion_timestamp:")
print(f"    Non-null rows : {ts_check['non_null_count']:,}")
print(f"    Range         : {ts_check['earliest']} → {ts_check['latest']}")

# Assembly distribution — shows GRCh37/GRCh38 split. Filtered in Silver
print("\n  Assembly distribution (top values):")
(
    df_check
    .groupBy("Assembly")
    .count()
    .orderBy("count", ascending=False)
    .show(5, truncate=False)
)

# ClinicalSignificance sample — shows the messiness Silver will clean
print("  ClinicalSignificance sample (10 distinct values):")
(
    df_check
    .select("ClinicalSignificance")
    .distinct()
    .limit(10)
    .show(truncate=False)
)

# Null rate for key columns
print("  Null counts for key columns:")
key_cols = ["GeneSymbol", "ClinicalSignificance", "PhenotypeList", 
            "Assembly", "Chromosome", "LastEvaluated"]

total = df_check.count()

null_counts = df_check.select([
    count(when(col(c).isNull() | (col(c) == ""), c)).alias(c)
    for c in key_cols
]).collect()[0]

for c in key_cols:
    pct = (null_counts[c] / total) * 100
    print(f"    {c:35s}: {null_counts[c]:>7,}  ({pct:.1f}%)")

print("\n" + "=" * 60)
print("  BRONZE COMPLETE — safe to run 02_silver_transform")
print("=" * 60)